# 08 - Sub-experiment 4: Error Analysis

This notebook compares the best LLM approach against the best classical model on the same test split and analyzes disagreement patterns.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys
sys.modules["tensorflow"] = None

### What this does and why

Aggregate metrics hide important behavior. Here we inspect where models disagree to understand which post styles favor the LLM and which favor classical embeddings + supervised learning.

In [2]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
root = Path.cwd()
outputs_dir = root / "outputs"
if not outputs_dir.exists():
    outputs_dir = root.parent / "outputs"

sample_df = pd.read_csv(outputs_dir / "llm_sample.csv")
cv = pd.read_csv(outputs_dir / "reddit_cv_summary.csv")
X_all = np.load(outputs_dir / "reddit_embeddings.npy")
y_all = np.load(outputs_dir / "reddit_labels.npy").astype(int)

if len(y_all) != X_all.shape[0]:
    raise ValueError("Embeddings and labels have mismatched lengths.")

best_prompt = pd.read_json(outputs_dir / "best_prompt.json", typ="series")
best_labels = [best_prompt["label_0_text"], best_prompt["label_1_text"]]
fewshot_meta = pd.read_json(outputs_dir / "fewshot_vs_zeroshot_meta.json", typ="series")
llm_method = fewshot_meta["winner"]

best_trained = cv.sort_values("f1_weighted_mean", ascending=False).iloc[0]
best_trained_name = best_trained["model"]

sample_row_ids = sample_df["row_id"].astype(int).to_numpy()
X_sample = X_all[sample_row_ids]
y_sample = y_all[sample_row_ids]

X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X_sample,
    y_sample,
    np.arange(len(sample_df)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_sample,
)

pu_path = root / "pipeline_utils.py"
if not pu_path.exists():
    pu_path = root / "major 2" / "pipeline_utils.py"
if not pu_path.exists():
    pu_path = root.parent / "major 2" / "pipeline_utils.py"
if not pu_path.exists():
    raise FileNotFoundError("Could not locate pipeline_utils.py")

spec = importlib.util.spec_from_file_location("pipeline_utils", pu_path)
pu = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pu)

models = pu.build_models()
classical_model = models[best_trained_name]
classical_model.fit(X_tr, y_tr)
classical_pred = classical_model.predict(X_te).astype(int)

few_shot_prefix = (
    "Example 1 (Mental Health Risk): \"i've been feeling really down lately and can't seem to find any motivation to do anything\"\n"
    "Example 2 (Mental Health Risk): \"the anxiety is getting worse and i don't know how to cope anymore\"\n"
    "Example 3 (High Risk Suicidal): \"i've been thinking about ending it all and have a plan ready for tonight\"\n"
    "Example 4 (High Risk Suicidal): \"i said goodbye to everyone today nobody knows why but i do\"\n"
    "Now classify the following post:\n\n"
)

test_texts = sample_df.iloc[idx_te]["text_clean"].astype(str).tolist()
llm_inputs = [few_shot_prefix + t for t in test_texts] if llm_method == "Few-Shot" else test_texts

try:
    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=-1,
    )
except Exception as e:
    raise RuntimeError(
        "Failed to load facebook/bart-large-mnli. Install transformers and torch. "
        f"Original error: {e}"
    )

llm_pred, llm_conf = [], []
for t in tqdm(llm_inputs, desc=f"LLM inference ({llm_method})"):
    out = classifier(t, candidate_labels=best_labels)
    winner = out["labels"][0]
    llm_pred.append(0 if winner == best_labels[0] else 1)
    llm_conf.append(float(out["scores"][0]))

rows = []
for i, (text, true_y, yp_llm, conf, yp_cls) in enumerate(zip(test_texts, y_te.astype(int), llm_pred, llm_conf, classical_pred)):
    llm_ok = int(yp_llm) == int(true_y)
    cls_ok = int(yp_cls) == int(true_y)
    if llm_ok and not cls_ok:
        category = "LLM advantage"
    elif cls_ok and not llm_ok:
        category = "Classical advantage"
    elif llm_ok and cls_ok:
        category = "Agreement correct"
    else:
        category = "Hard cases"

    rows.append({
        "test_row": i,
        "category": category,
        "text": text,
        "text_trunc": (text[:150] + "...") if len(text) > 150 else text,
        "true_label": int(true_y),
        "llm_pred": int(yp_llm),
        "llm_confidence": float(conf),
        "classical_pred": int(yp_cls),
        "classical_model": best_trained_name,
        "llm_method": llm_method,
    })

full_df = pd.DataFrame(rows)
full_df.to_csv(outputs_dir / "error_analysis_full.csv", index=False)

summary = full_df["category"].value_counts().rename_axis("category").reset_index(name="count")
summary["pct"] = 100 * summary["count"] / summary["count"].sum()
summary.to_csv(outputs_dir / "error_analysis_summary.csv", index=False)

print("Best classical model:", best_trained_name)
print("Best LLM method:", llm_method)
print(summary)
for cat in ["LLM advantage", "Classical advantage", "Agreement correct", "Hard cases"]:
    sub = full_df[full_df["category"] == cat].head(5)
    print("\n===", cat, f"({len(full_df[full_df['category']==cat])} total)", "===")
    for _, r in sub.iterrows():
        print(f"- text: {r['text_trunc']}")
        print(f"  true={r['true_label']} llm={r['llm_pred']} (conf={r['llm_confidence']:.3f}) classical={r['classical_pred']}")

summary

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
LLM inference (Zero-Shot): 100%|██████████| 40/40 [01:53<00:00,  2.85s/it]

Best classical model: SVM
Best LLM method: Zero-Shot
              category  count   pct
0    Agreement correct     21  52.5
1        LLM advantage     10  25.0
2  Classical advantage      7  17.5
3           Hard cases      2   5.0

=== LLM advantage (10 total) ===
- text: there was once a boy who was never frowning. he always laughed. he was always the one who made jokes. but when he got home, he would lay in his bed an...
  true=1 llm=1 (conf=0.678) classical=0
- text: better yet, there's going to be a psychiatrist convention on april 25-29 in philadelphia. go there, bring a gun, and open fire!
  true=1 llm=1 (conf=0.654) classical=0
- text: i had a traumatic past... including getting buried alive by my step father as a kid. ive been working through that trauma but some of the issues its c...
  true=0 llm=0 (conf=0.898) classical=1
- text: got so frustrated and triggered this morning that i resorted to my toxic outlet again. its an ongoing cycle... amp; x200b; the day i take my life

,category,count,pct
0,Agreement correct,21,52.5
1,LLM advantage,10,25.0
2,Classical advantage,7,17.5
3,Hard cases,2,5.0


## Pattern Analysis Notes

Use the printed category samples above to document your qualitative findings for reviewer discussion:
- Which language patterns favor the LLM?
- Which patterns favor the classical model?
- What makes hard cases difficult for both?